In [1]:
import os
import json
import argparse
import torch
import torchaudio
import numpy as np
import editdistance
from pathlib import Path
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)
import tgt

In [2]:
sampa_to_api_single = {
    # vowels
    'a': 'a',      # (ignore 'ɑ' duplicate → keep simple)
    'e': 'e',
    'i': 'i',
    'o': 'o',
    'u': 'u',
    'y': 'y',

    '2': 'ø',
    '9': '9',      # œ
    '@': 'ə',
    'E': 'ɛ',
    'O': 'ɔ',

    # nasal vowels
    'a~': '@',     # ɑ̃
    'e~': '5',     # ɛ̃
    '9~': '1',     # œ̃ / ɛ̃
    'o~': '§',     # ɔ̃

    # consonants
    'b': 'b',
    'd': 'd',
    'f': 'f',
    'g': 'ɡ',
    'k': 'k',
    'l': 'l',
    'm': 'm',
    'n': 'n',
    'n=':'n',
    'p': 'p',
    's': 's',
    't': 't',
    'v': 'v',
    'w': 'w',
    'z': 'z',
    'j': 'j',

    # special consonants
    'R': 'ʁ',
    'N': 'ŋ',
    'H': 'ɥ',

    # palatal / special
    'J': 'ɲ',      # prefer ɲ over ɟ (French consistent)
    'S': 'ʃ',      # prefer fricative over affricate
    'Z=': 'dʒ',

    # affricates (if explicitly needed)
    's': 'ts',
    'Z': 'dʒ',

    # special cases from your table
    'm=': 'm',

    # silence / noise
    '_': '_',
    'spn': 'spn',
    'unk': 'spn',
    "%":'spn',
    "?":"spn",
    "0":"spn"
}

In [3]:
def map_ref_to_api_single(ph):
     
    if ph in sampa_to_api_single:
        mapped = sampa_to_api_single[ph]
        return mapped
    else:
        if ph!= None and ph != "":
            return ph
def read_textgrid(tg_path, tier_name="phone"):

    tg        = tgt.io.read_textgrid(tg_path)
    tier      = tg.get_tier_by_name(tier_name)
    silence   = {"", "SIL", "sil", "spn", "SP", "<SIL>", "_","0","fe~","sjo~","Ra~"}
    phonemes = []
    for iv in tier.intervals:
        label = iv.text.strip()
        phoneme=map_ref_to_api_single(label)
        if phoneme in silence:
            continue
        phonemes.append(phoneme)
    return phonemes



def ctc_collapse(token_ids: list, blank_id: int) -> list:
    """
    Standard CTC greedy decode:
    remove consecutive duplicates then remove blank tokens.
    """
    collapsed, prev = [], None
    for t in token_ids:
        if t != prev:
            if t != blank_id:
                collapsed.append(t)
        prev = t
    return collapsed



In [10]:
from VAD_chunk import *

def evaluate_file(
    wav_path:  str,
    tg_path:   str,
    tier_name: str,
    model:     Wav2Vec2ForCTC,
    processor: Wav2Vec2Processor,
    device:    torch.device,
) -> dict:
    """
    Evaluate one (wav, textgrid) pair using VAD chunking.
    Each chunk is decoded independently, then phoneme sequences
    are concatenated and compared against the full reference.
    """
    # ── Load and preprocess audio ────────────────────────────────────
    audio, sr = sf.read(wav_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)          # stereo → mono
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
 
    wav      = torch.from_numpy(audio.astype(np.float32))
    blank_id = processor.tokenizer.pad_token_id  # [PAD] = 0 = CTC blank
 
    # ── VAD chunking ─────────────────────────────────────────────────
    chunks = vad_chunk_with_timestamps(wav)
 
    if not chunks:
        # Fallback: whole file as one chunk
        chunks = [{"start": 0.0, "end": wav.shape[0] / 16000}]
 
    # ── Decode each chunk, concatenate phonemes ───────────────────────
    hyp = []
    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_audio  = wav[start_sample:end_sample]
 
        # Skip chunks that are too short for the model
        if chunk_audio.shape[0] < 400:   # < 25ms
            continue
 
        inputs = processor(
            chunk_audio.numpy(),
            sampling_rate  = 16000,
            return_tensors = "pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
 
        with torch.no_grad():
            logits = model(**inputs).logits   # (1, T, vocab)
 
        pred_ids  = logits.argmax(dim=-1)[0].tolist()
        collapsed = ctc_collapse(pred_ids, blank_id)
        chunk_hyp = [
            processor.tokenizer.convert_ids_to_tokens(i)
            for i in collapsed
        ]
        hyp.extend(chunk_hyp)
 
    # ── Reference from TextGrid ──────────────────────────────────────
    ref = read_textgrid(tg_path, tier_name)
 
    # ── PER ──────────────────────────────────────────────────────────
    n_errors = editdistance.eval(hyp, ref)
    n_ref    = max(len(ref), 1)
    per      = n_errors / n_ref
 
    return {
        "file":     os.path.basename(wav_path),
        "hyp":      hyp,
        "ref":      ref,
        "per":      per,
        "n_ref":    len(ref),
        "n_errors": n_errors,
        "n_chunks": len(chunks),
    }

 


In [11]:
model_path = "results/w2vCTC_joint_nofxfy/checkpoint-21092"
vocab_path = "vocab_w2vCTC.json"
wav_dir = "/vol/corpora/Rhapsodie/wav16k_corrected"
tg_dir = "/vol/corpora/Rhapsodie/TextGrids-fev2013/" 
tg_tier ="phone"
output = "results/per_evaluation.txt"
style_csv = "/vol/corpora/Rhapsodie/wav_style.csv"

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from types import SimpleNamespace
class GumbelQuantizerEMA(nn.Module):
    def __init__(self, input_dim: int, num_vars: int = 320,
                 temp: float = 2.0, decay: float = 0.99):
        super().__init__()
        self.num_vars = num_vars
        self.temp     = temp
        self.decay    = decay

        self.weight_proj = nn.Linear(input_dim, num_vars)

        self.register_buffer("codevectors", torch.randn(num_vars, input_dim))
        self.register_buffer("cluster_size", torch.ones(num_vars))
        self.register_buffer("ema_embed",   torch.randn(num_vars, input_dim))
        nn.init.uniform_(self.codevectors, -1.0, 1.0)

    def forward(self, X: torch.Tensor):
        B, T, D = X.shape
        logits   = self.weight_proj(X)

        if self.training:
            probs = F.gumbel_softmax(logits, tau=self.temp, hard=True)

            with torch.no_grad():
                flat_probs = probs.reshape(-1, self.num_vars)
                flat_X     = X.reshape(-1, D)
                counts     = flat_probs.sum(0)
                embed_sum  = flat_probs.T @ flat_X

                self.cluster_size = (self.decay * self.cluster_size
                                     + (1 - self.decay) * counts)
                self.ema_embed    = (self.decay * self.ema_embed
                                     + (1 - self.decay) * embed_sum)
                self.codevectors  = (self.ema_embed
                                     / self.cluster_size.unsqueeze(1).clamp(min=1e-5))
        else:
            indices = logits.argmax(dim=-1)
            probs   = F.one_hot(indices, self.num_vars).float()

        quantized = torch.matmul(probs, self.codevectors)
        return quantized, probs
# =========================
# Step 9: Model
# =========================

class Wav2Vec2ForCTC_FS_REC(nn.Module):
    def __init__(self, base_model, vocab_size, hidden_dim,
                 ctc_tokenizer, n_negatives=50, temperature=0.1):
        super().__init__()
        self.wav2vec2            = base_model.wav2vec2
        self.ctc_head            = base_model.lm_head
        self.ctc_tokenizer       = ctc_tokenizer
        self.n_negatives         = n_negatives
        self.temperature         = temperature
        self.align_temperature   = 5.0                              # ← must match training
        self.fx                  = nn.Linear(hidden_dim, hidden_dim, bias=False)  # ← no bias
        self.quantizer           = GumbelQuantizerEMA(input_dim=hidden_dim)
        self.reconstruction_head = nn.Linear(2 * hidden_dim, hidden_dim)

    def _ctc_decode_batch(self, logits):
        pred_ids  = logits.argmax(dim=-1)
        sequences = []
        for seq in pred_ids:
            unique_mask = torch.cat([
                torch.tensor([True], device=seq.device),
                seq[1:] != seq[:-1]
            ])
            collapsed = seq[unique_mask]
            collapsed = collapsed[collapsed != 0]
            if collapsed.numel() == 0:
                collapsed = torch.tensor([1], device=seq.device)
            sequences.append(collapsed)
        return torch.nn.utils.rnn.pad_sequence(
            sequences, batch_first=True, padding_value=0
        )

    def compute_phoneme_embeddings_from_ctc(self, X, logits, labels_clean):
        B, T, D   = X.shape
        ctc_probs = logits.softmax(dim=-1)
        Y_emb_list = []
        for b in range(B):
            phone_embs = []
            for p_id in labels_clean[b]:
                p_id = p_id.item()
                if p_id == 0:
                    phone_embs.append(torch.zeros(D, device=X.device))
                    continue
                weights = ctc_probs[b, :, p_id]
                weight_sum = weights.sum()
                if weight_sum < 1e-6:
                    weights = torch.ones_like(weights) / weights.shape[0]
                else:
                    weights = weights / weight_sum
                emb = (weights.unsqueeze(-1) * X[b]).sum(0)
                phone_embs.append(emb)
            Y_emb_list.append(torch.stack(phone_embs))
        return torch.stack(Y_emb_list)

    def compute_alignment(self, X, Y_emb):
        B, T, D = X.shape
        _, N, _ = Y_emb.shape
        X_proj  = self.fx(X)
        X_norm  = F.normalize(X_proj, dim=-1)
        Y_norm  = F.normalize(Y_emb,  dim=-1)
        D_mat   = self.align_temperature * torch.matmul(
            Y_norm, X_norm.transpose(1, 2)
        )
        n_idx = torch.arange(N, device=X.device).float() / max(N - 1, 1)
        t_idx = torch.arange(T, device=X.device).float() / max(T - 1, 1)
        prior = -10.0 * (n_idx.unsqueeze(1) - t_idx.unsqueeze(0)) ** 2
        D_mat = D_mat + prior.unsqueeze(0)
        return torch.softmax(D_mat, dim=1)

    def forward(self, input_values, attention_mask=None):
        outputs      = self.wav2vec2(input_values, attention_mask=attention_mask)
        X            = outputs.last_hidden_state
        logits       = self.ctc_head(X)
    
        with torch.no_grad():
            labels_clean = self._ctc_decode_batch(logits)
    
        Y_emb = self.compute_phoneme_embeddings_from_ctc(
            X.detach(), logits.detach(), labels_clean
        )
        A = self.compute_alignment(X, Y_emb)
    
        # Return object with .logits so evaluate_per.py works unchanged
        return SimpleNamespace(
            logits       = logits,
            alignment    = A,
            labels_clean = labels_clean,
        )


In [32]:
device = "cpu"
print(f"Device: {device}")

# ── Processor ────────────────────────────────────────────────────────────
print(f"Loading processor from {vocab_path} ...")
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file           = vocab_path,
    unk_token            = "[UNK]",
    pad_token            = "[PAD]",
    word_delimiter_token = "",
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size          = 1,
    sampling_rate         = 16000,
    padding_value         = 0.0,
    do_normalize          = True,
    return_attention_mask = True,
)
processor = Wav2Vec2Processor(
    feature_extractor = feature_extractor,
    tokenizer         = tokenizer,
)

# ── Model ─────────────────────────────────────────────────────────────────
print(f"Loading model from {model_path} ...")
checkpoint = "results/w2vCTC_joint_nofxfy/checkpoint-21092"
ctc_checkpoint = "results/w2vCTC/checkpoint-23430"
# ── Model ─────────────────────────────────────────────────────────────
print(f"Loading CTC architecture from {ctc_checkpoint}...")
base_model = Wav2Vec2ForCTC.from_pretrained(
    ctc_checkpoint,
    ctc_loss_reduction = "mean",
    ctc_zero_infinity  = True,
    pad_token_id       = tokenizer.pad_token_id,
    vocab_size         = len(tokenizer),
).to(device)

model = Wav2Vec2ForCTC_FS_REC(
    base_model,
    vocab_size = len(processor.tokenizer),
    hidden_dim = base_model.config.hidden_size,
    ctc_tokenizer = processor.tokenizer,
).to(device)
model.wav2vec2.feature_extractor._freeze_parameters()

bin_path = os.path.join(checkpoint, "pytorch_model.bin")
sft_path = os.path.join(checkpoint, "model.safetensors")
if os.path.exists(bin_path):
    state_dict = torch.load(bin_path, map_location="cpu")
elif os.path.exists(sft_path):
    from safetensors.torch import load_file
    state_dict = load_file(sft_path)
else:
    raise FileNotFoundError(
        f"No model weights found in {checkpoint}\n"
        f"Expected pytorch_model.bin or model.safetensors"
    )
model.eval()
print(f"Vocab size: {len(processor.tokenizer)}")


Device: cpu
Loading processor from vocab_w2vCTC.json ...
Loading model from results/w2vCTC_joint_nofxfy/checkpoint-21092 ...
Loading CTC architecture from results/w2vCTC/checkpoint-23430...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Vocab size: 42


In [33]:
import csv
def load_style_map(csv_path: str, file_col: str, style_col: str) -> dict:
    """
    Load a CSV mapping filename → style.
    Returns dict: {stem: style}
    Handles filenames with or without .wav extension.
    """
    style_map = {}
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if file_col not in reader.fieldnames:
            raise ValueError(
                f"Column '{file_col}' not found in {csv_path}.\n"
                f"Available columns: {reader.fieldnames}"
            )
        if style_col not in reader.fieldnames:
            raise ValueError(
                f"Column '{style_col}' not found in {csv_path}.\n"
                f"Available columns: {reader.fieldnames}"
            )
        for row in reader:
            fname = row[file_col].strip()
            # normalize to stem (no extension)
            stem  = Path(fname).stem
            style = row[style_col].strip()
            style_map[stem] = style
    return style_map
 


In [34]:

# ── Style map ─────────────────────────────────────────────────────────────
style_map = {}
print(f"Loading style map from {style_csv} ...")
style_map = load_style_map(style_csv, "file", "style")
styles    = set(style_map.values())
print(f"Found {len(styles)} styles: {sorted(styles)}")
print(f"Style map covers {len(style_map)} files\n")

# ── Collect file pairs ────────────────────────────────────────────────────
wav_dir   = Path(wav_dir)
tg_dir    = Path(tg_dir)
wav_files = sorted(wav_dir.glob("*.wav"))

pairs = []
for wav_path in wav_files:
    tg_path = tg_dir / (wav_path.stem + "-Pro.TextGrid")

    if tg_path.exists():
        pairs.append((str(wav_path), str(tg_path)))
print(f"Evaluating {len(pairs)} file pairs ...\n")



Loading style map from /vol/corpora/Rhapsodie/wav_style.csv ...
Found 3 styles: ['planned', 'semi', 'spont']
Style map covers 57 files

Evaluating 54 file pairs ...



In [35]:
style_map1={}
for i in style_map.keys():
    style_map1["Rhap-"+i]=style_map[i]

In [36]:
# ── Evaluate file by file ─────────────────────────────────────────────────
results      = []
total_errors = 0
total_ref    = 0
failed       = []

for i, (wav_path, tg_path) in enumerate(pairs):
    try:
        r = evaluate_file(
            wav_path, tg_path, tg_tier,
            model, processor, device,
        )
        # Attach style if available
        stem       = Path(wav_path).stem
        r["style"] = style_map1.get(stem, "unknown")

        results.append(r)
        total_errors += r["n_errors"]
        total_ref    += r["n_ref"]

        if (i + 1) % 50 == 0 or (i + 1) == len(pairs):
            running_per = total_errors / max(total_ref, 1)
            print(
                f"  [{i+1:4d}/{len(pairs)}]  "
                f"running PER = {running_per*100:.2f}%"
            )

    except Exception as e:
        failed.append((os.path.basename(wav_path), str(e)))
        print(f"  FAILED: {os.path.basename(wav_path)} — {e}")



  [  50/54]  running PER = 21.56%
  [  54/54]  running PER = 20.82%


In [47]:
results[0]["ref"]

['ø',
 'i',
 'l',
 'z',
 'a',
 'v',
 'ɛ',
 'dʒ',
 'a',
 'v',
 'ɛ',
 'd',
 'l',
 'a',
 'ʃ',
 '@',
 'ts',
 'd',
 'a',
 'v',
 'w',
 'a',
 'ʁ',
 'd',
 'e',
 'z',
 '@',
 'f',
 '@',
 'k',
 'i',
 't',
 'ʁ',
 'a',
 'v',
 'a',
 'j',
 'ɛ',
 'ø',
 'dʒ',
 'ə',
 'd',
 'i',
 'ʁ',
 'ɛ',
 'ts',
 'ts',
 'e',
 't',
 'ɛ',
 'i',
 'l',
 'p',
 'ʁ',
 'e',
 'f',
 'e',
 'ʁ',
 'ɛ',
 'ʁ',
 'i',
 'ɡ',
 'ɔ',
 'l',
 'e',
 'k',
 'ə',
 'd',
 't',
 'ʁ',
 'a',
 'v',
 'a',
 'j',
 'e',
 'i',
 'l',
 'z',
 'e',
 't',
 'ɛ',
 't',
 'u',
 't',
 'a',
 'f',
 'ɛ',
 'n',
 'ɔ',
 'ʁ',
 'm',
 'o',
 'm',
 'ɛ',
 'dʒ',
 'v',
 'ø',
 'd',
 'i',
 'ʁ',
 'i',
 'l',
 'z',
 '§',
 't',
 'ʁ',
 'a',
 'v',
 'a',
 'j',
 'e',
 'n',
 'ɔ',
 'ʁ',
 'm',
 'a',
 'l',
 'm',
 '@',
 'e',
 'ts',
 'e',
 'v',
 'ʁ',
 'ɛ',
 'k',
 'ə',
 'b',
 '§',
 'a',
 'b',
 'i',
 't',
 '@',
 'd',
 '@',
 'l',
 'ts',
 '@',
 't',
 'ʁ',
 'd',
 'ə',
 'p',
 'a',
 'ʁ',
 'i',
 'ø',
 'l',
 'e',
 'z',
 'e',
 'k',
 'ɔ',
 'l',
 'ts',
 '§',
 'd',
 'ə',
 't',
 'ʁ',
 'ɛ',
 'b',
 '§',
 'n',
 

In [46]:
results[0]["hyp"]

['e',
 'ʒ',
 'a',
 'v',
 'ɛ',
 'l',
 'a',
 'ʃ',
 '@',
 's',
 'd',
 'a',
 'v',
 'w',
 'a',
 'ʁ',
 'd',
 'e',
 'z',
 '@',
 'f',
 '@',
 'k',
 'i',
 't',
 'ʁ',
 'a',
 'v',
 'a',
 'j',
 'ɛ',
 't',
 '9',
 'ʒ',
 'ə',
 'd',
 'i',
 'ʁ',
 'ɛ',
 'i',
 'p',
 'ʁ',
 'e',
 'f',
 'e',
 'ʁ',
 'ɛ',
 'ʁ',
 'i',
 'ɡ',
 'o',
 'l',
 'e',
 'k',
 'ə',
 'd',
 'ə',
 't',
 'ʁ',
 'a',
 'v',
 'a',
 'j',
 'e',
 'i',
 'l',
 'z',
 'e',
 't',
 'ɛ',
 't',
 'u',
 't',
 'a',
 'f',
 'ɛ',
 'n',
 'ɔ',
 'ʁ',
 'm',
 'o',
 'm',
 'ɛ',
 'ʒ',
 'ə',
 'v',
 'ø',
 'd',
 'i',
 'ʁ',
 'i',
 'l',
 'z',
 '§',
 't',
 'ʁ',
 'a',
 'v',
 'a',
 'j',
 'e',
 'n',
 'ɔ',
 'ʁ',
 'm',
 'a',
 'l',
 'm',
 '@',
 'e',
 's',
 'ɛ',
 'v',
 'ʁ',
 'ɛ',
 'k',
 'ə',
 'b',
 '§',
 'n',
 'a',
 'b',
 'i',
 't',
 '@',
 'd',
 '@',
 'l',
 'ə',
 's',
 '@',
 't',
 'ʁ',
 'd',
 'ə',
 'p',
 'a',
 'ʁ',
 'i',
 '9',
 'z',
 'e',
 'k',
 'ɔ',
 'l',
 's',
 '§',
 'o',
 'k',
 '1',
 'p',
 'ʁ',
 'ɔ',
 'b',
 'l',
 'ɛ',
 'm',
 'm',
 'w',
 'a',
 'ʒ',
 'e',
 'y',
 'o',
 'k',
 '1',
 'p'

In [37]:
from pathlib import Path
from collections import defaultdict

# ── Per-style summary ─────────────────────────────────────────────────────
style_stats = None
if style_map:
    style_errors = defaultdict(int)
    style_ref    = defaultdict(int)
    style_files  = defaultdict(int)

    for r in results:
        s = r["style"]
        style_errors[s] += r["n_errors"]
        style_ref[s]    += r["n_ref"]
        style_files[s]  += 1

    style_stats = {}
    for s in sorted(style_errors.keys()):
        style_stats[s] = {
            "per":   style_errors[s] / max(style_ref[s], 1),
            "files": style_files[s],
            "refs":  style_ref[s],
            "errors": style_errors[s],
        }

    print(f"\n{'='*55}")
    print(f"  PER BY STYLE")
    print(f"{'='*55}")
    for s, v in style_stats.items():
        print(
            f"  {s:20s}  PER={v['per']*100:5.2f}%  "
            f"files={v['files']:4d}  phones={v['refs']:6d}"
        )
    print(f"{'='*55}\n")





  PER BY STYLE
  planned               PER=16.46%  files=  11  phones= 29234
  semi                  PER=23.87%  files=   4  phones= 14697
  spont                 PER=22.51%  files=  39  phones= 48964



In [38]:
# ── Summary ───────────────────────────────────────────────────────────────
global_per = total_errors / max(total_ref, 1)
mean_per   = float(np.mean([r["per"] for r in results])) if results else 0.0

print(f"\n{'='*55}")
print(f"  Files evaluated : {len(results)}")
print(f"  Files failed    : {len(failed)}")
print(f"  Total phonemes  : {total_ref}")
print(f"  Total errors    : {total_errors}")
print(f"  Global PER      : {global_per*100:.2f}%")
print(f"  Mean file PER   : {mean_per*100:.2f}%")
print(f"{'='*55}\n")

# ── Save detailed output ──────────────────────────────────────────────────
if output:
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    with open(output, "w") as f:
        f.write(f"Global PER : {global_per*100:.2f}%\n")
        f.write(f"Mean PER   : {mean_per*100:.2f}%\n")
        f.write(f"Files      : {len(results)}\n")
        f.write(f"Failed     : {len(failed)}\n")
        f.write("="*55 + "\n\n")

        for r in sorted(results, key=lambda x: -x["per"]):
            f.write(f"File : {r['file']}\n")
            f.write(
                f"PER  : {r['per']*100:.2f}%  "
                f"({r['n_errors']} errors / {r['n_ref']} phones)\n"
            )
            f.write(f"REF  : {' '.join(r['ref'])}\n")
            f.write(f"HYP  : {' '.join(r['hyp'])}\n\n")

        if failed:
            f.write("="*55 + "\nFAILED\n" + "="*55 + "\n")
            for fname, err in failed:
                f.write(f"{fname}: {err}\n")

        if style_stats:
            f.write("\n" + "="*55 + "\nPER BY STYLE\n" + "="*55 + "\n")
            for s, v in style_stats.items():
                f.write(
                    f"{s:20s}  PER={v['per']*100:.2f}%  "
                    f"files={v['files']}  phones={v['refs']}  "
                    f"errors={v['errors']}\n"
                )

    print(f"Detailed results saved to: {output}")


  Files evaluated : 54
  Files failed    : 0
  Total phonemes  : 92895
  Total errors    : 19345
  Global PER      : 20.82%
  Mean file PER   : 19.64%

Detailed results saved to: results/per_evaluation.txt
